# Tropeiro Intel · Official Colab

**Campaign Intelligence · Attribution Engineering · Actionable Intelligence**

Official public machine for **Tropeiro Intel 3.1.1**.

> **PASSIVE_DEFAULT:** direct HTTP probing is disabled by default. Optional providers fail closed: missing credentials become `SKIPPED`, not notebook crashes.

Read results in this order: **Executive Assessment → Action Matrix → IOC Decisions → Attribution → Collection Gaps → Next Best Pivots → Evidence Ledger**.

In [ ]:
#@title 01) Install / update Tropeiro Intel
import os, subprocess, sys
REPO="/content/tropeiro-intel"
if not os.path.exists(os.path.join(REPO,".git")):
    subprocess.run(["git","clone","-q","https://github.com/Ridd1kulusC0d3r/tropeiro-intel.git",REPO],check=True)
else:
    subprocess.run(["git","-C",REPO,"pull","-q","--ff-only"],check=False)
subprocess.run([sys.executable,"-m","pip","install","-q","-e",f"{REPO}[colab]"],check=True)
print("Tropeiro Intel ready.")

In [ ]:
#@title 02) Imports, helpers and optional secrets
from datetime import datetime, timezone, timedelta
from pathlib import Path
import os, json, re
import pandas as pd

from tropeiro.models import Observation, now_iso
from tropeiro.utils import extract_iocs, hostname, root_domain
from tropeiro.config import Settings
from tropeiro.collectors import dns, rdap, crtsh, history, urlscan, threatintel
from tropeiro.evidence.ledger import build as build_ledger
from tropeiro.correlation.graph import CampaignGraph
from tropeiro.attribution.engine import AttributionEngine
from tropeiro.intelligence.warninglists import WarningListEngine
from tropeiro.intelligence.decision_objects import build_ioc_decisions
from tropeiro.intelligence.collection_gap import build_collection_gaps
from tropeiro.intelligence.pivot import rank_pivots
from tropeiro.intelligence.action_matrix import build_action_matrix
from tropeiro.intelligence.confidence import multidimensional_confidence
from tropeiro.intelligence.lifecycle import infer_campaign_lifecycle, infrastructure_churn
from tropeiro.intelligence.victimology import infer_victimology
from tropeiro.intelligence.objectives import infer_objectives
from tropeiro.intelligence.detection import build_detection_package
from tropeiro.intelligence.briefing import build_intelligence_brief
from tropeiro.reporting.rich_html import build_report
from tropeiro.reporting.exporter import export_selected, zip_exports

def get_secret(name):
    try:
        from google.colab import userdata
        value=userdata.get(name)
        if value: return value
    except Exception:
        pass
    return os.getenv(name,"")

SECRETS={k:get_secret(k) for k in [
    "URLSCAN_API_KEY","VT_API_KEY","THREATFOX_AUTH_KEY",
    "DNSDUMPSTER_API_KEY","FOFA_API_KEY","CENSYS_PAT","CENSYS_ORG_ID"
]}
SOURCE_STATUS=[]
def safe_call(source,fn,*args,**kwargs):
    try:
        value=fn(*args,**kwargs)
        SOURCE_STATUS.append({"source":source,"status":"OK","error":""})
        return value
    except Exception as e:
        SOURCE_STATUS.append({"source":source,"status":"UNAVAILABLE","error":str(e)[:300]})
        return None

print("Optional providers configured:",[k for k,v in SECRETS.items() if v] or "none")

In [ ]:
#@title 03) Case configuration
CASE_ID="TI-DEMO-001" #@param {type:"string"}
ANALYST="Analyst" #@param {type:"string"}
BRAND="" #@param {type:"string"}
SEEDS="example.com" #@param {type:"string"}
LURE_TEXT="" #@param {type:"string"}
ENABLE_WAYBACK=True #@param {type:"boolean"}
ENABLE_COMMONCRAWL=False #@param {type:"boolean"}
ENABLE_URLSCAN=True #@param {type:"boolean"}
ENABLE_OTX=True #@param {type:"boolean"}
ENABLE_VT=False #@param {type:"boolean"}
ENABLE_THREATFOX=False #@param {type:"boolean"}
ENABLE_HTTP_PROBE=False #@param {type:"boolean"}

if ENABLE_HTTP_PROBE:
    raise ValueError("The public Colab keeps direct HTTP probing disabled. Use authorized local workflows for active validation.")

WORKSPACE=Path(f"/content/tropeiro_{re.sub(r'[^A-Za-z0-9_.-]+','_',CASE_ID)}")
WORKSPACE.mkdir(parents=True,exist_ok=True)
SETTINGS=Settings(case_id=CASE_ID,analyst=ANALYST,brand=BRAND,workspace=WORKSPACE)
OBS=[]
def observe(entity,entity_type,source,value,confidence="observed",notes=""):
    OBS.append(Observation(str(entity),str(entity_type),str(source),str(value),confidence=confidence,notes=str(notes)))

seed_text="\n".join([SEEDS,LURE_TEXT])
IOCS=extract_iocs(seed_text)
for raw in [x.strip() for x in re.split(r"[\n,]+",SEEDS) if x.strip()]:
    if raw.startswith(("http://","https://")):
        IOCS.setdefault("url",[]).append(raw)
        IOCS.setdefault("domain",[]).append(hostname(raw))
    elif "." in raw:
        IOCS.setdefault("domain",[]).append(hostname(raw))
for k in list(IOCS): IOCS[k]=sorted(set(x for x in IOCS[k] if x))
DOMAINS=sorted(set(root_domain(x) for x in IOCS.get("domain",[]) if root_domain(x)))
print("Case:",CASE_ID,"| domains:",DOMAINS)

In [ ]:
#@title 04) Passive collection: DNS, RDAP and Certificate Transparency
OWNERSHIP={}
for domain in DOMAINS:
    OWNERSHIP[domain]={"domain":domain,"dns":{},"rdap":{},"cert_names":[]}
    for rtype in ["A","AAAA","MX","NS"]:
        vals=safe_call(f"dns:{rtype}",dns.query,domain,rtype) or []
        OWNERSHIP[domain]["dns"][rtype]=vals
        for value in vals: observe(domain,"domain",f"dns:{rtype}",value)
    r=safe_call("rdap",rdap.lookup,domain) or {}
    OWNERSHIP[domain]["rdap"]=r
    for org in r.get("registrant_orgs",[]): observe(domain,"domain","rdap:registrant_org",org)
    for org in r.get("registrar_orgs",[]): observe(domain,"domain","rdap:registrar_org",org)
    for ns in r.get("nameservers",[]): observe(domain,"domain","rdap:nameserver",ns)
    ct=safe_call("crt.sh",crtsh.lookup,domain) or []
    names=set()
    for row in ct if isinstance(ct,list) else []:
        for name in str(row.get("name_value","")).splitlines():
            name=name.strip().lower().lstrip("*.")
            if name: names.add(name)
    OWNERSHIP[domain]["cert_names"]=sorted(names)[:1000]
    for name in sorted(names)[:1000]: observe(domain,"domain","crt.sh",name)
print("Observations:",len(OBS))

In [ ]:
#@title 05) Passive historical and threat-intelligence enrichment
URLSCAN_ROWS=[]; OTX_ROWS=[]; WAYBACK_ROWS=[]; CC_ROWS=[]; VT_ROWS=[]; THREATFOX_ROWS=[]
for domain in DOMAINS:
    if ENABLE_WAYBACK:
        rows=safe_call("wayback",history.wayback,domain,1000) or []
        WAYBACK_ROWS += [{"domain":domain,"row":r} for r in rows]
        for r in rows[:1000]:
            if len(r)>=2: observe(domain,"domain","wayback",str(r[0]),notes=str(r[1]))
    if ENABLE_COMMONCRAWL:
        rows=safe_call("commoncrawl",history.commoncrawl,domain,500) or []
        CC_ROWS += [{"domain":domain,**r} for r in rows if isinstance(r,dict)]
        for r in rows[:500]: observe(domain,"domain","commoncrawl",str(r.get("url",r)))
    if ENABLE_URLSCAN:
        data=safe_call("urlscan",urlscan.search,domain,SECRETS["URLSCAN_API_KEY"]) or {}
        rows=data.get("results",[]) if isinstance(data,dict) else []
        URLSCAN_ROWS += rows
        for r in rows[:200]:
            page=r.get("page",{}); task=r.get("task",{})
            u=page.get("url") or task.get("url")
            if u: observe(domain,"domain","urlscan",u)
    if ENABLE_OTX:
        data=safe_call("otx",threatintel.otx_domain,domain) or {}
        rows=data.get("url_list",[]) if isinstance(data,dict) else []
        OTX_ROWS += rows
        for r in rows[:300]:
            u=r.get("url") if isinstance(r,dict) else None
            if u: observe(domain,"domain","otx",u)
    if ENABLE_VT:
        if SECRETS["VT_API_KEY"]:
            data=safe_call("virustotal",threatintel.virustotal_domain,domain,SECRETS["VT_API_KEY"])
            if data: VT_ROWS.append(data)
        else: SOURCE_STATUS.append({"source":"virustotal","status":"SKIPPED","error":"missing VT_API_KEY"})
    if ENABLE_THREATFOX:
        if SECRETS["THREATFOX_AUTH_KEY"]:
            data=safe_call("threatfox",threatintel.threatfox_search,domain,SECRETS["THREATFOX_AUTH_KEY"])
            if data: THREATFOX_ROWS.append(data)
        else: SOURCE_STATUS.append({"source":"threatfox","status":"SKIPPED","error":"missing THREATFOX_AUTH_KEY"})

display(pd.DataFrame(SOURCE_STATUS).drop_duplicates())

In [ ]:
#@title 06) Evidence Ledger, relationship graph and campaign clusters
LEDGER_DF=build_ledger(OBS,SETTINGS.source_reliability)
G=CampaignGraph()
for domain,data in OWNERSHIP.items():
    G.add_node(domain,"domain")
    for rtype,vals in data.get("dns",{}).items():
        for value in vals:
            target=value.lower().rstrip(".") if isinstance(value,str) else str(value)
            G.add_rel(domain,target,f"dns_{rtype.lower()}",f"dns:{rtype}",weight=.45 if rtype in {"A","AAAA"} else .55)
    for org in data.get("rdap",{}).get("registrant_orgs",[]):
        G.add_rel(domain,"org:"+org,"registrant_org","rdap",weight=.90)
    for org in data.get("rdap",{}).get("registrar_orgs",[]):
        G.add_rel(domain,"registrar:"+org,"registered_by","rdap",weight=.30)
    for name in data.get("cert_names",[])[:300]:
        if name!=domain: G.add_rel(domain,name,"certificate_name_overlap","crt.sh",weight=.55)

CAMPAIGN_CLUSTERS=[
    {"cluster_id":f"CAMP-{i:03d}","size":len(nodes),"nodes":sorted(map(str,nodes))[:200]}
    for i,nodes in enumerate(G.clusters(),1)
]
print("Evidence:",len(LEDGER_DF),"| graph nodes:",G.g.number_of_nodes(),"| edges:",G.g.number_of_edges())

In [ ]:
#@title 07) Attribution and competing hypotheses
ATTRIBUTIONS=[]
if len(DOMAINS)>=2:
    engine=AttributionEngine()
    for i,left in enumerate(DOMAINS):
        for right in DOMAINS[i+1:]:
            ev=[]; l=OWNERSHIP[left]; r=OWNERSHIP[right]
            if set(l.get("rdap",{}).get("registrant_orgs",[])) & set(r.get("rdap",{}).get("registrant_orgs",[])):
                ev.append({"code":"same_registrant_org","source_family":"registration","source_reliability":.95})
            if set(l.get("dns",{}).get("NS",[])) & set(r.get("dns",{}).get("NS",[])):
                ev.append({"code":"same_ns_pair","source_family":"dns","source_reliability":.95})
            if set(l.get("dns",{}).get("MX",[])) & set(r.get("dns",{}).get("MX",[])):
                ev.append({"code":"same_mx","source_family":"dns","source_reliability":.95})
            if set(l.get("dns",{}).get("A",[])) & set(r.get("dns",{}).get("A",[])):
                ev.append({"code":"same_ip","source_family":"hosting","source_reliability":.85})
            if ev: ATTRIBUTIONS.append(engine.assess(left,right,ev))
display(pd.DataFrame(ATTRIBUTIONS))

In [ ]:
#@title 08) Decision intelligence: lifecycle, IOCs, gaps, pivots and actions
LIFECYCLE=infer_campaign_lifecycle([{"observed_at":o.observed_at} for o in OBS])
VICTIMOLOGY=infer_victimology(LURE_TEXT,BRAND,IOCS.get("url",[]),[])
OBJECTIVES=infer_objectives(LURE_TEXT,IOCS.get("url",[]))
CHURN=infrastructure_churn([o.__dict__ for o in OBS])
warning=WarningListEngine()
recent_cutoff=datetime.now(timezone.utc)-timedelta(days=7)
ioc_candidates=[]
for d in DOMAINS:
    rows=[o for o in OBS if o.entity==d]
    families=len(set(o.source.split(":")[0] for o in rows))
    recent=False
    for o in rows:
        try:
            recent |= datetime.fromisoformat(o.observed_at.replace("Z","+00:00")).astimezone(timezone.utc)>=recent_cutoff
        except Exception: pass
    support=min(.78,.22+min(.32,len(rows)*.025)+min(.20,families*.05))
    if any((r.get("page",{}).get("domain") or "").lower()==d for r in URLSCAN_ROWS): support=min(.92,support+.12)
    ioc_candidates.append({"value":d,"type":"domain","confidence":support,"active":bool(recent),
                           "evidence_count":len(rows),"source_families":families,"context":{},
                           "campaign":CASE_ID,"rationale":["passive evidence only"]})
IOC_DECISIONS=build_ioc_decisions(ioc_candidates,warning)
families=len(set(o.source.split(":")[0] for o in OBS))
context={
 "active_state_known":any(x.get("active_now") for x in IOC_DECISIONS),
 "registration_entity_known":any(v.get("rdap",{}).get("registrant_orgs") for v in OWNERSHIP.values()),
 "redirect_chain_known":False,"credential_endpoint_known":False,
 "victimology_known":VICTIMOLOGY.get("confidence") not in {"LOW","INSUFFICIENT"},
 "operator_identity_known":any(a.get("identity_attribution")=="SUPPORTED" for a in ATTRIBUTIONS),
 "independent_source_families":families
}
COLLECTION_GAPS=build_collection_gaps(context)
PIVOT_CANDIDATES=[]
for d,data in OWNERSHIP.items():
    for org in data.get("rdap",{}).get("registrant_orgs",[]):
        PIVOT_CANDIDATES.append({"value":org,"type":"public_contact","evidence_value":.9,"novelty":.8,"source_reliability":.95,"expected_noise":.25})
    for ns in data.get("dns",{}).get("NS",[]):
        PIVOT_CANDIDATES.append({"value":ns,"type":"nameserver","evidence_value":.65,"novelty":.55,"source_reliability":.95,"expected_noise":.65})
    for cert in data.get("cert_names",[])[:30]:
        PIVOT_CANDIDATES.append({"value":cert,"type":"certificate","evidence_value":.75,"novelty":.7,"source_reliability":.9,"expected_noise":.4})
PIVOTS=rank_pivots(PIVOT_CANDIDATES)
operator_score=max([a.get("common_operator_score",0) for a in ATTRIBUTIONS] or [0])
malicious_support=max([x.get("confidence",0) for x in IOC_DECISIONS] or [0])
CONFIDENCE=multidimensional_confidence(
 infrastructure_relationship=.85 if G.g.number_of_edges() else .2,
 common_operator=operator_score, malicious_intent=malicious_support,
 campaign_active=.75 if any(x.get("active_now") for x in IOC_DECISIONS) else .3,
 victimology=.65 if VICTIMOLOGY.get("confidence")=="MODERATE" else .3,
 identity_attribution=0.0)
DETECTION_PACKAGE=build_detection_package(IOC_DECISIONS,BRAND)
ACTION_MATRIX=build_action_matrix(IOC_DECISIONS,COLLECTION_GAPS,DETECTION_PACKAGE)
display(pd.DataFrame(ACTION_MATRIX).head(20))

In [ ]:
#@title 09) Executive assessment and enriched local report
EXECUTIVE_ASSESSMENT={
 "judgment":f"Tropeiro Intel assessment for {CASE_ID}",
 "implication":"Review P1 actions, validate collection gaps and avoid treating shared infrastructure as operator identity.",
 "confidence_note":"Confidence dimensions are separate; investigation priority is not a maliciousness probability."
}
INTEL_BRIEF=build_intelligence_brief(CASE_ID,EXECUTIVE_ASSESSMENT,ACTION_MATRIX,COLLECTION_GAPS,PIVOTS,LIFECYCLE,CONFIDENCE,VICTIMOLOGY,OBJECTIVES,changes={"infrastructure_churn":CHURN})
REPORT_DATA={
 "meta":{"case_id":CASE_ID,"analyst":ANALYST,"brand":BRAND,"generated_at":now_iso(),"tool":"Tropeiro Intel","version":"3.1.1"},
 "executive_brief":INTEL_BRIEF,"executive_assessment":EXECUTIVE_ASSESSMENT,
 "action_matrix":ACTION_MATRIX,"ioc_decisions":IOC_DECISIONS,
 "collection_gaps":COLLECTION_GAPS,"next_best_pivots":PIVOTS,"confidence":CONFIDENCE,
 "victimology":VICTIMOLOGY,"objectives":OBJECTIVES,"lifecycle":LIFECYCLE,
 "attribution_assessments":ATTRIBUTIONS,"campaign_clusters":CAMPAIGN_CLUSTERS,
 "evidence_ledger":LEDGER_DF.to_dict("records") if not LEDGER_DF.empty else [],
 "source_status":SOURCE_STATUS,"ownership":OWNERSHIP,"detection_package":DETECTION_PACKAGE
}
REPORT_PATH=WORKSPACE/"Tropeiro_Intel_Report.html"
build_report(REPORT_DATA,REPORT_PATH)
print(json.dumps(INTEL_BRIEF,ensure_ascii=False,indent=2,default=str)[:10000])
print("Report:",REPORT_PATH)
try:
    from IPython.display import display, IFrame
    display(IFrame(src=str(REPORT_PATH),width="100%",height=900))
except Exception: pass

In [ ]:
#@title 10) Export Center
import ipywidgets as widgets
from IPython.display import display, clear_output
choices={
 "Local HTML report":"report_html","Executive Brief JSON":"executive_brief_json",
 "IOC Decisions CSV":"ioc_decisions_csv","Action Matrix CSV":"action_matrix_csv",
 "Collection Gaps CSV":"collection_gaps_csv","Next Best Pivots CSV":"next_best_pivots_csv",
 "Evidence Ledger CSV":"evidence_ledger_csv","Attribution CSV":"attribution_csv",
 "Detection Package JSON":"detection_json","Full Case JSON":"case_json"
}
checks={label:widgets.Checkbox(value=(key in {"report_html","executive_brief_json","ioc_decisions_csv","action_matrix_csv"}),description=label) for label,key in choices.items()}
button=widgets.Button(description="Generate selected export")
out=widgets.Output()
def _export(_):
    with out:
        clear_output()
        selected={choices[label] for label,w in checks.items() if w.value}
        files=export_selected(REPORT_DATA,WORKSPACE/"export",selected,REPORT_PATH)
        zp=zip_exports(files,WORKSPACE/f"Tropeiro_{CASE_ID}_Export.zip")
        print("ZIP:",zp)
        try:
            from google.colab import files as colab_files
            colab_files.download(str(zp))
        except Exception: pass
button.on_click(_export)
display(widgets.VBox(list(checks.values())+[button,out]))

## Interpretation guardrails

- A missing provider result is **not proof of benignity**.
- A shared IP, ASN, registrar or provider is weak evidence unless stronger independent signals converge.
- Operator correlation is not real-world identity attribution.
- BLOCK recommendations require analyst review, especially for shared infrastructure.
- Preserve the Evidence Ledger, source status and export manifest with the case record.